# 01 | 3C-O25 quick start

This notebook processes **one above-water radiometric spectrum** with the 3C-O25 model. It is designed for readers who may be new to Python or Jupyter notebooks.

By the end, you will have:

- loaded the supplied measurements
- inspected the three radiometric signals
- fitted the 3C-O25 model
- separated the measured signal into aquatic reflectance and glint
- checked the quality of the fit
- interpreted the output without over-interpreting the fitted parameters

## Scientific idea

Above-water total radiance contains both the water-leaving signal and light reflected by the air-water surface. In radiance-reflectance units, the model separates the measurement as

$$
\frac{L_t}{E_s} = R_{rs} + R_g,
$$

where:

- $L_t$ is total radiance measured by the downward-looking sensor
- $L_i$ is sky radiance measured toward the sky
- $E_s$ is downwelling planar irradiance
- $R_{rs}$ is remote-sensing reflectance
- $R_g$ is the modeled glint contribution

The fit uses an analytical aquatic reflectance, $R_{rs,mod}$, together with the glint model to reproduce the measured $L_t/E_s$. The final reported aquatic spectrum is obtained from the measurement:

$$
R_{rs} = \frac{L_t}{E_s} - R_g.
$$

This distinction is important. The analytical $R_{rs,mod}$ supports the fit, while the final $R_{rs}$ preserves fine spectral features in the measured signal.

**Scientific reference:** Pitarch, J. (2026), *A general model for sun and sky glint removal in above-water optical radiometry: mathematical description and Python code*, Earth Science Informatics, 19:78. DOI: 10.1007/s12145-026-02114-w.

## 1. Before you begin 🧭

This notebook works in **JupyterLab**, **Jupyter Notebook**, and **Visual Studio Code** with the Jupyter extension. The calculations and results are the same in all three environments.

Run the notebook cells from top to bottom. For a complete reproducibility check, restart the kernel and run all cells.

### 1.1 Create the project environment

From the repository root, create a virtual environment and install the dependencies used by both notebooks.

#### Windows PowerShell

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -e ".[examples,timeseries]"
```

#### Linux and macOS

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e ".[examples,timeseries]"
```

### 1.2 Choose a notebook interface

Use any one of the following options.

#### JupyterLab

Install JupyterLab in the active environment if necessary, then start it from the repository root:

```bash
python -m pip install jupyterlab
python -m jupyter lab
```

Open this notebook from the `notebooks/` directory.

#### Jupyter Notebook

Install the classic Notebook interface in the active environment if necessary, then start it from the repository root:

```bash
python -m pip install notebook
python -m jupyter notebook
```

Open this notebook from the `notebooks/` directory.

#### Visual Studio Code

1. Open the repository folder in Visual Studio Code.
2. Install the Microsoft Jupyter extension if it is not already installed.
3. Open this notebook from the `notebooks/` directory.
4. Select the Python interpreter under `.venv` as the notebook kernel.
5. Choose **Restart Kernel and Run All Cells**.

> **Kernel check:** whichever interface you use, make sure the selected notebook kernel belongs to `.venv`. If `.venv` does not appear in the kernel list, restart the notebook interface after activating the environment.


In [ ]:
import sys
from pathlib import Path


def find_repository_root(start: Path) -> Path:
    """Find the nearest parent containing the project files."""
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "pyproject.toml").is_file() and (folder / "src" / "rrs3c").is_dir():
            return folder
    raise FileNotFoundError(
        "Could not locate the 3C-Rrs-O25 repository. "
        "Keep this notebook inside the repository and try again."
    )


REPO_ROOT = find_repository_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
DATA_DIR = REPO_ROOT / "data"
EXAMPLE_FILE = REPO_ROOT / "examples" / "example_single_spectrum.csv"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_ROOT)
print("Example data:", EXAMPLE_FILE)
print("Ancillary data:", DATA_DIR)
assert EXAMPLE_FILE.is_file(), "The example spectrum is missing."
assert DATA_DIR.is_dir(), "The ancillary data directory is missing."
print("Setup check passed.")

## 2. Import the required packages

If this cell reports a missing package, return to the setup instructions above and make sure the notebook is using the `.venv` kernel.

In [ ]:
import lmfit as lm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rrs3c import rrs_model_3C_O25

print("Imports successful.")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

## 3. Load the supplied spectrum

The CSV contains 15 introductory lines followed by four columns:

1. wavelength in nanometres
2. sky radiance, $L_i$
3. total radiance, $L_t$
4. downwelling irradiance, $E_s$

The radiance and irradiance values share compatible power and area units, so the ratios $L_i/E_s$ and $L_t/E_s$ have units of $sr^{-1}$.

In [ ]:
measurements = pd.read_csv(
    EXAMPLE_FILE,
    skiprows=15,
    index_col=0,
)

measurements.index = measurements.index.astype(float)
measurements.index.name = "wavelength_nm"
measurements.columns = ["Li", "Lt", "Es"]

wl = measurements.index.to_numpy(dtype=float)
Li = measurements["Li"].to_numpy(dtype=float)
Lt = measurements["Lt"].to_numpy(dtype=float)
Es = measurements["Es"].to_numpy(dtype=float)

print(f"Loaded {wl.size} wavelengths from {wl.min():.0f} to {wl.max():.0f} nm.")
display(measurements.head())

## 4. Inspect the measured signals

Plotting the inputs before fitting is a useful quality-control step. The three signals have different magnitudes, so they are shown in separate panels.

Look for:

- gaps or non-finite values
- abrupt single-band spikes
- irradiance values at or below zero
- unexpected wavelength limits

The supplied file should contain finite data on a regular wavelength grid.

In [ ]:
assert np.all(np.isfinite(wl)), "Wavelength contains non-finite values."
assert np.all(np.diff(wl) > 0), "Wavelength must be strictly increasing."
assert np.all(np.isfinite(Li)), "Li contains non-finite values."
assert np.all(np.isfinite(Lt)), "Lt contains non-finite values."
assert np.all(np.isfinite(Es)), "Es contains non-finite values."
assert np.all(Es > 0), "Es must be positive before radiance ratios are calculated."

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
axes[0].plot(wl, Es, color="tab:orange")
axes[0].set_ylabel("Es (W/m²/nm)")
axes[0].set_title("Measured radiometric inputs")

axes[1].plot(wl, Li, color="tab:blue")
axes[1].set_ylabel("Li (W/m²/sr/nm)")

axes[2].plot(wl, Lt, color="tab:green")
axes[2].set_ylabel("Lt (W/m²/sr/nm)")
axes[2].set_xlabel("Wavelength (nm)")

for axis in axes:
    axis.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 5. Build the model inputs

The model is fitted in radiance-reflectance units:

$$
L_i/E_s \quad \text{and} \quad L_t/E_s.
$$

The example geometry is:

- solar zenith angle $\theta_s = 59^\circ$
- viewing zenith angle $\theta_v = 35^\circ$
- relative azimuth $\Delta\phi = 100^\circ$

Geometry matters because the O25 aquatic model explicitly accounts for sun-view geometry, including relative azimuth. The atmospheric ancillary tuple contains air-mass type, relative humidity, and pressure.

The weighting array excludes 760 to 765 nm, where a strong atmospheric water-vapour feature can disturb the fit, and gives additional influence to wavelengths above 800 nm in this example.

In [ ]:
LiEs = Li / Es
LtEs = Lt / Es

geometry = (59.0, 35.0, 100.0)  # theta_s, theta_v, relative azimuth (all in degrees)
ancillary = (4.0, 60.0, 1013.25)  # air mass, relative humidity (%), pressure (hPa)

weights = np.ones_like(wl, dtype=float)
weights[(wl >= 760) & (wl <= 765)] = 0.0
weights[wl > 800] = 5.0

fig, axis = plt.subplots(figsize=(9, 4))
axis.plot(wl, LiEs, label="Li / Es")
axis.plot(wl, LtEs, label="Lt / Es")
axis.set_xlabel("Wavelength (nm)")
axis.set_ylabel("Radiance reflectance (sr$^{-1}$)")
axis.grid(True, alpha=0.3)
axis.legend()
fig.tight_layout()
plt.show()

## 6. Define the fitting parameters

Each `lmfit` parameter has an initial value, a flag that determines whether the parameter is varied, and lower and upper bounds.

The parameters serve different parts of the model:

- `C`, `N`, and `Y` drive the aquatic bio-optical model
- `SNAP` and `Sg` control absorption spectral slopes
- `alpha` and `beta` describe aerosol spectral behaviour
- `rho`, `rho_d`, and `rho_s` scale glint components

> **Interpretation warning:** optimized aquatic parameters are fitting variables, not validated constituent retrievals. Optical ambiguity means different parameter combinations can generate very similar reflectance spectra. Dedicated algorithms should be applied to the final $R_{rs}$ when constituent concentrations are required.

In [ ]:
params = lm.Parameters()
params.add_many(
    ("C", 5.0, True, 0.1, 50.0, None),
    ("N", 1.0, True, 0.01, 100.0, None),
    ("Y", 0.5, True, 0.01, 5.0, None),
    ("SNAP", 0.015, True, 0.005, 0.03, None),
    ("Sg", 0.015, True, 0.005, 0.03, None),
    ("rho", 0.02, False, 0.0, 0.03, None),
    ("rho_d", 0.0, True, 0.0, 10.0, None),
    ("rho_s", 0.0, True, -0.01, 0.01, None),
    ("alpha", 0.2, True, 0.0, 2.0, None),
    ("beta", 0.05, True, 0.0, 1.0, None),
)

parameter_overview = pd.DataFrame(
    {
        "initial": [parameter.value for parameter in params.values()],
        "vary": [parameter.vary for parameter in params.values()],
        "minimum": [parameter.min for parameter in params.values()],
        "maximum": [parameter.max for parameter in params.values()],
    },
    index=list(params.keys()),
)
parameter_overview

## 7. Fit the 3C-O25 model

The model object loads ancillary optical data and prepares geometry interpolators. Create it once, outside loops over spectra.

The optimizer seeks a combination of aquatic, atmospheric, and surface parameters for which

$$
\left.\frac{L_t}{E_s}\right|_{mod} = R_{rs,mod} + R_g
$$

closely matches the measured $L_t/E_s$.

In [ ]:
model = rrs_model_3C_O25(data_folder=DATA_DIR)

result, Rrs_mod, Rg = model.fit_LtEs(
    wl=wl,
    LiEs=LiEs,
    LtEs=LtEs,
    params=params,
    weights=weights,
    geom=geometry,
    anc=ancillary,
    method="leastsq",
    verbose=False,
)

LtEs_mod = Rrs_mod + Rg
Rrs_output = LtEs - Rg
residual = LtEs_mod - LtEs
rmse = float(np.sqrt(np.mean(residual**2)))

print("Fit success:", getattr(result, "success", "not reported"))
print("Optimizer message:", getattr(result, "message", "not reported"))
print("Function evaluations:", getattr(result, "nfev", "not reported"))
print(f"RMSE in Lt/Es space: {rmse:.6g} sr^-1")

## 8. Inspect the signal separation

The first plot checks whether the modeled total signal follows the measurement. The second plot shows the modeled glint contribution and the final aquatic reflectance.

A tight total-signal fit is a necessary practical quality indicator because the glint estimate is inferred through that fit. It is not, by itself, proof of a perfect separation or a complete uncertainty estimate.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axes[0].plot(wl, LtEs, color="black", linewidth=1.5, label="Lt / Es measured")
axes[0].plot(wl, LtEs_mod, color="tab:red", linestyle="--", label="Lt / Es modeled")
axes[0].set_ylabel("Radiance reflectance (sr$^{-1}$)")
axes[0].set_title("Model-to-data agreement")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(wl, Rg, color="tab:green", label="Rg modeled glint")
axes[1].plot(wl, Rrs_output, color="tab:blue", label="Rrs output = Lt/Es - Rg")
axes[1].plot(wl, Rrs_mod, color="tab:purple", linestyle=":", label="Rrs modeled fitting term")
axes[1].set_xlabel("Wavelength (nm)")
axes[1].set_ylabel("Reflectance (sr$^{-1}$)")
axes[1].set_title("Separated components")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.tight_layout()
plt.show()

## 9. Examine residuals and fitted parameters

Residuals reveal where the combined aquatic-plus-glint model does not reproduce the measured total signal. Fine scale differences are inevitable, which are due to the difference in phytoplankton community composition between the reality and the model. Still, the model must be able to broadly reproduce the  measurements. Generally, we may find:

- coherent spectral structure --> sign of model success
- a broad offset across many wavelengths --> sign of model failure
- large deviations around atmospheric absorption bands --> inevitable but inoffensive
- isolated spikes that may indicate measurement artefacts --> reassess the measurements

The fitted parameters are shown for transparency, but the aquatic values should not be interpreted as direct concentration retrievals.

In [ ]:
fig, axis = plt.subplots(figsize=(9, 4))
axis.axhline(0.0, color="black", linewidth=0.8)
axis.plot(wl, residual, color="tab:red")
axis.set_xlabel("Wavelength (nm)")
axis.set_ylabel("Modeled minus measured Lt/Es (sr$^{-1}$)")
axis.set_title(f"Fit residual, RMSE = {rmse:.3g} sr$^{{-1}}$")
axis.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

fitted_parameters = pd.DataFrame(
    {
        "value": [parameter.value for parameter in result.params.values()],
        "varied": [parameter.vary for parameter in result.params.values()],
        "at_lower_bound": [
            np.isfinite(parameter.min) and np.isclose(parameter.value, parameter.min)
            for parameter in result.params.values()
        ],
        "at_upper_bound": [
            np.isfinite(parameter.max) and np.isclose(parameter.value, parameter.max)
            for parameter in result.params.values()
        ],
    },
    index=list(result.params.keys()),
)
fitted_parameters

## 10. A compact scientific interpretation

Use the following hierarchy when interpreting this example:

1. **Did the optimizer finish successfully?** A failed optimization invalidates the result.
2. **Does modeled $L_t/E_s$ follow measured $L_t/E_s$?** Structured residuals indicate that the fitted components do not fully explain the measurement.
3. **Is $R_g$ carrying the expected glint burden?** Glint can be large and spectrally structured, especially when direct sunlight contributes.
4. **Is the final $R_{rs}$ spectrally plausible?** Check for discontinuities, strong negative regions, and unexplained atmospheric features.
5. **Are parameters stuck at bounds?** This can signal insufficient constraints, a poor starting point, model-to-data mismatch, or parameter ambiguity.

The 3C-O25 method explicitly represents diffuse and direct glint components and uses an aquatic model that accounts for relative azimuth. This is particularly relevant outside traditional low-glint geometries. Nevertheless, trusted quality-control procedures and independent validation remain necessary.

The model's fine-scale output is `Rrs_output`, calculated from the measurement after subtracting `Rg`. `Rrs_mod` is the smoother analytical aquatic term used to guide the optimization.

## 11. Save the result, if desired

This cell is optional. Set `SAVE_OUTPUT = True` to create a CSV under `examples/output/`. The output directory is created automatically.

In [ ]:
SAVE_OUTPUT = False

if SAVE_OUTPUT:
    output_folder = REPO_ROOT / "examples" / "output"
    output_folder.mkdir(parents=True, exist_ok=True)
    output_file = output_folder / "quickstart_single_spectrum_output.csv"

    output = pd.DataFrame(
        {
            "wavelength_nm": wl,
            "LtEs_measured": LtEs,
            "LtEs_modeled": LtEs_mod,
            "Rrs_output": Rrs_output,
            "Rrs_modeled": Rrs_mod,
            "Rg_modeled": Rg,
            "residual": residual,
            "weight": weights,
        }
    )
    output.to_csv(output_file, index=False)
    print("Saved:", output_file)
else:
    print("Output was not saved. Set SAVE_OUTPUT = True to write the CSV.")

## 12. Where to go next

- Read `examples/README.md` for an overview of all examples.
- Continue with `02_TimeSeries_Processing.ipynb` to process repeated measurements.
- Use `scripts/run_3C.py` for non-interactive command-line processing.
- Read the scientific paper for the model equations, assumptions, performance comparison, and limitations.
- Apply dedicated bio-optical retrieval algorithms to the final $R_{rs}$ when water constituent concentrations are required.

### Key cautions

- A successful fit is necessary but not sufficient evidence of an accurate glint correction.
- Optimized aquatic parameters are not direct geophysical retrievals.
- Parameter bounds and initial values should reflect the measurement environment.
- Low-glint geometry remains desirable even though 3C-O25 can model challenging geometries.
- Whitecaps, foam, bubbles, polarization effects, and unusual atmospheric or high-altitude conditions can remain outside the model assumptions.

You have now completed a full single-spectrum 3C-O25 processing workflow.